# lltPretrainData data prep (CPU)

The funnel on one Common Crawl WARC file, the quality classifier, and the training corpora for the GPU kernel.

In [ ]:

import os, platform, subprocess, sys, time
print("python", platform.python_version(), "| cpus", os.cpu_count())
# One package per call, so one failed build cannot block the rest. fasttext-numpy2 ships cp313 wheels.
for pkg in ["trafilatura", "datasketch", "warcio", "fasttext-numpy2", "datatrove", "spacy"]:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], capture_output=True, text=True)
    print(pkg, "exit", r.returncode, r.stderr[-400:])
for m in ["trafilatura", "datasketch", "warcio", "fasttext", "datatrove", "spacy", "tokenizers", "sklearn", "numpy", "pandas"]:
    try:
        mod = __import__(m); print(m, getattr(mod, "__version__", "?"))
    except Exception as e:
        print(m, "MISSING", repr(e)[:120])


In [ ]:
PTD_SRC = '"""lltPretrainData shared library: the filters, PII scrub, decontamination and MinHash dedup the lab\npage builds step by step. Plain Python and NumPy, so the same functions run in the notebook, in the\nKaggle data-prep kernel and (the NumPy parts) in the browser.\n\nRule thresholds follow Rae et al. 2021 (Gopher, Appendix A, Table A1) and Raffel et al. 2020 (C4,\nsection 2.2), with the defaults datatrove 0.10.1 uses for the same rules. Words are split on\nwhitespace here; datatrove splits with a language tokenizer, so a few borderline documents differ\n(the measured agreement is in the Run Library file lltPretrainData-cpu-funnel).\n"""\nimport hashlib\nimport re\nfrom collections import Counter\n\nimport numpy as np\n\n# ----------------------------------------------------------------------------------------------\n# Gopher quality rules (Rae et al. 2021, Appendix A)\nSTOP_WORDS = {"the", "be", "to", "of", "and", "that", "have", "with"}\nPUNCT = set("!\\"#$%&\'()*+,-./:;<=>?@[]^_`{|}~")\n\n\ndef gopher_quality(text):\n    """Return None if the document passes, else the name of the first rule it fails."""\n    words = text.split()\n    if not words:\n        return "empty"\n    content = [w for w in words if any(c not in PUNCT for c in w)]\n    if len(content) < 50:\n        return "too_few_words"\n    if len(content) > 100_000:\n        return "too_many_words"\n    mean_len = sum(len(w) for w in content) / len(content)\n    if mean_len < 3 or mean_len > 10:\n        return "mean_word_length"\n    if text.count("#") / len(words) > 0.1:\n        return "hash_ratio"\n    if (text.count("...") + text.count("…")) / len(words) > 0.1:\n        return "ellipsis_ratio"\n    lines = text.splitlines() or [text]\n    if sum(l.lstrip().startswith(("•", "-")) for l in lines) / len(lines) > 0.9:\n        return "bullet_lines"\n    if sum(l.rstrip().endswith(("...", "…")) for l in lines) / len(lines) > 0.3:\n        return "ellipsis_lines"\n    if sum(any(c.isalpha() for c in w) for w in words) / len(words) < 0.8:\n        return "alpha_words"\n    if len(STOP_WORDS & set(w.lower() for w in words)) < 2:\n        return "stop_words"\n    return None\n\n\n# ----------------------------------------------------------------------------------------------\n# Gopher repetition rules (Rae et al. 2021, Table A1)\nTOP_NGRAM = ((2, 0.20), (3, 0.18), (4, 0.16))\nDUP_NGRAM = ((5, 0.15), (6, 0.14), (7, 0.13), (8, 0.12), (9, 0.11), (10, 0.10))\n\n\ndef _dups(items):\n    seen, n_dup, chars = set(), 0, 0\n    for x in items:\n        if x in seen:\n            n_dup += 1\n            chars += len(x)\n        else:\n            seen.add(x)\n    return n_dup, chars\n\n\ndef _dup_ngram_chars(words, n):\n    # Characters covered by n-grams already seen, skipping ahead past each repeat (as datatrove does).\n    seen, chars, i = set(), 0, 0\n    while i < len(words) - n + 1:\n        g = " ".join(words[i:i + n])\n        if g in seen:\n            chars += sum(len(w) for w in words[i:i + n])\n            i += n\n        else:\n            seen.add(g)\n            i += 1\n    return chars\n\n\ndef gopher_repetition(text):\n    if not text.strip():\n        return "empty"\n    paras = re.split("\\n{2,}", text.strip())\n    n_dup, chars = _dups(paras)\n    if n_dup / len(paras) > 0.30:\n        return "dup_paragraphs"\n    if chars / len(text) > 0.20:\n        return "dup_paragraph_chars"\n    lines = re.split("\\n+", text)\n    n_dup, chars = _dups(lines)\n    if n_dup / len(lines) > 0.30:\n        return "dup_lines"\n    if chars / len(text) > 0.20:\n        return "dup_line_chars"\n    words = text.split()\n    for n, frac in TOP_NGRAM:\n        grams = [" ".join(words[i:i + n]) for i in range(len(words) - n + 1)]\n        if grams:\n            g, c = Counter(grams).most_common(1)[0]\n            if len(g) * c / len(text) > frac:\n                return "top_%d_gram" % n\n    for n, frac in DUP_NGRAM:\n        if _dup_ngram_chars(words, n) / len(text) > frac:\n            return "dup_%d_gram" % n\n    return None\n\n\n# ----------------------------------------------------------------------------------------------\n# C4 rules (Raffel et al. 2020, section 2.2), as the paper states them: keep lines that end in\n# terminal punctuation and have at least 5 words, drop pages with fewer than 3 sentences.\n# datatrove 0.10.1\'s C4QualityFilter defaults swap those two numbers (5 sentences, 3 words).\n# The bad-words list is not applied here.\nPOLICY = ("terms of use", "privacy policy", "cookie policy", "uses cookies", "use of cookies", "use cookies")\nCITATION = re.compile(r"\\[\\d*\\]|\\[edit\\]|\\[citation needed\\]")\nSENT_END = re.compile(r"[.!?][\\"\')\\]]*(?=\\s|$)")\n\n\ndef c4_clean(text, terminal_punct=True):\n    """Return (cleaned_text, None) if the page is kept, else (None, rule)."""\n    kept, sentences = [], 0\n    for line in text.splitlines():\n        line = line.strip()\n        words = line.split()\n        if any(len(w) > 1000 for w in words):\n            continue\n        line = CITATION.sub("", line)\n        if terminal_punct and (not line.endswith((".", "?", "!", \'"\', "\'")) or line.endswith("...")):\n            continue\n        if len(words) < 5:\n            continue\n        low = line.lower()\n        if "lorem ipsum" in low:\n            return None, "lorem_ipsum"\n        if "javascript" in low:\n            continue\n        if "{" in line:\n            return None, "curly_bracket"\n        if any(p in low for p in POLICY):\n            continue\n        sentences += max(1, len(SENT_END.findall(line)))\n        kept.append(line)\n    if sentences < 3:\n        return None, "fewer_than_3_sentences"\n    return "\\n".join(kept), None\n\n\n# ----------------------------------------------------------------------------------------------\n# PII scrub: email addresses and public IPv4 addresses, replaced by fixed placeholders.\nEMAIL = re.compile(r"\\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}\\b")\nIPV4 = re.compile(r"\\b(?:(?:25[0-5]|2[0-4][0-9]|1[0-9][0-9]|[1-9]?[0-9])\\.){3}(?:25[0-5]|2[0-4][0-9]|1[0-9][0-9]|[1-9]?[0-9])\\b")\n\n\ndef _private_ip(ip):\n    a, b = (int(x) for x in ip.split(".")[:2])\n    return a in (0, 10, 127) or (a == 192 and b == 168) or (a == 172 and 16 <= b <= 31) or (a == 169 and b == 254) or a >= 224\n\n\ndef scrub_pii(text):\n    """Return (text, n_emails, n_ips) with emails and public IPv4s replaced."""\n    text, n_email = EMAIL.subn("<EMAIL>", text)\n    n_ip = 0\n\n    def ip_sub(m):\n        nonlocal n_ip\n        if _private_ip(m.group(0)):\n            return m.group(0)\n        n_ip += 1\n        return "<IP>"\n\n    text = IPV4.sub(ip_sub, text)\n    return text, n_email, n_ip\n\n\n# ----------------------------------------------------------------------------------------------\n# n-gram decontamination: drop a document that shares any n-gram (13 words, lower-cased) with an\n# evaluation set.\nWORD = re.compile(r"[a-z0-9]+")\n\n\ndef ngrams(text, n=13):\n    w = WORD.findall(text.lower())\n    return {" ".join(w[i:i + n]) for i in range(len(w) - n + 1)}\n\n\ndef build_eval_index(eval_texts, n=13):\n    idx = set()\n    for t in eval_texts:\n        idx |= ngrams(t, n)\n    return idx\n\n\ndef contaminated(text, index, n=13):\n    return not ngrams(text, n).isdisjoint(index)\n\n\n# ----------------------------------------------------------------------------------------------\n# Exact dedup on normalised text\ndef norm_text(text):\n    return " ".join(WORD.findall(text.lower()))\n\n\ndef exact_key(text):\n    return hashlib.sha1(norm_text(text).encode()).hexdigest()\n\n\n# ----------------------------------------------------------------------------------------------\n# MinHash + LSH in NumPy. Shingles are word 5-grams of the normalised text, hashed to 31 bits.\n# Each of the K hash functions is h(x) = (a * x + b) mod P with P = 2**31 - 1, so a * x stays\n# below 2**62 and fits in uint64.\nP31 = np.uint64(2**31 - 1)\n\n\ndef shingles(text, n=5):\n    w = norm_text(text).split()\n    grams = {" ".join(w[i:i + n]) for i in range(max(1, len(w) - n + 1))} if w else set()\n    return np.array([int.from_bytes(hashlib.blake2b(g.encode(), digest_size=4).digest(), "little") & 0x7FFFFFFF\n                     for g in grams], dtype=np.uint64)\n\n\ndef make_hashes(k, seed=1):\n    rng = np.random.default_rng(seed)\n    a = rng.integers(1, 2**31 - 1, size=k, dtype=np.uint64)\n    b = rng.integers(0, 2**31 - 1, size=k, dtype=np.uint64)\n    return a, b\n\n\ndef minhash(sh, a, b):\n    if sh.size == 0:\n        return np.full(a.size, 2**31 - 1, dtype=np.uint64)\n    return ((sh[:, None] * a[None, :] + b[None, :]) % P31).min(axis=0)\n\n\ndef lsh_candidates(sigs, bands, rows):\n    """sigs: (n_docs, bands*rows). Return the set of (i, j) pairs that share at least one band."""\n    pairs = set()\n    for bnd in range(bands):\n        buckets = {}\n        for i, s in enumerate(sigs[:, bnd * rows:(bnd + 1) * rows]):\n            buckets.setdefault(s.tobytes(), []).append(i)\n        for ids in buckets.values():\n            for x in range(len(ids)):\n                for y in range(x + 1, len(ids)):\n                    pairs.add((ids[x], ids[y]))\n    return pairs\n\n\ndef p_candidate(j, bands, rows):\n    """Probability that two documents with Jaccard similarity j share at least one band."""\n    return 1 - (1 - j ** rows) ** bands\n\n\ndef clusters(n, pairs):\n    """Union-find over candidate pairs. Returns the root of each document; keep one per root."""\n    parent = list(range(n))\n\n    def find(x):\n        while parent[x] != x:\n            parent[x] = parent[parent[x]]\n            x = parent[x]\n        return x\n\n    for i, j in pairs:\n        ri, rj = find(i), find(j)\n        if ri != rj:\n            parent[max(ri, rj)] = min(ri, rj)\n    return [find(i) for i in range(n)]\n\n\ndef jaccard(s1, s2):\n    s1, s2 = set(s1.tolist()), set(s2.tolist())\n    return len(s1 & s2) / max(1, len(s1 | s2))\n'
open('/tmp/ptd.py', 'w').write(PTD_SRC)
PREP_SRC = '"""lltPretrainData data prep. Runs on a Kaggle CPU kernel (llt-ptd-prep, 4 cores, no GPU quota).\n\nPart A, the funnel. One Common Crawl WARC file (CC-MAIN-2024-10, segment file 00000) through\nextraction, language ID, Gopher quality, Gopher repetition, C4 rules, exact and MinHash dedup,\nbenchmark decontamination and the PII scrub, with the document count after every stage and\nremoved examples. Cross-checks: datatrove\'s filters on the same documents, datasketch\'s MinHash LSH\non the same documents, and the matching WET file\'s text for the same URLs.\n\nPart B, the quality classifier. A hashed bag-of-words ridge regression trained on the FineWeb-Edu\nLlama 3 annotations (one parquet shard), scored on a held-out split.\n\nPart C, the training corpora. C4 en.noclean shard 00000 (CC WET text with a language filter only)\nas the unfiltered corpus U, the same documents through our filters as F, the top half of F by\nclassifier score as E, plus wikitext-103 and TinyStories for the mixture runs. Tokenized with one\n4,096-token byte-level BPE trained on a sample of all three domains. Also writes the canaries, the\nheld-out evaluation sets and the train/val leakage split for the GPU kernel.\n\nUsage: python prep_ptd.py <out_dir> [--small]\n"""\nimport gzip, hashlib, io, json, os, random, re, signal, sys, time, urllib.request\nfrom multiprocessing import Pool\n\nimport numpy as np\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, HERE)\nimport ptd  # noqa: E402\n\nOUT = sys.argv[1]\nSMALL = "--small" in sys.argv\nRAW = os.environ.get("PTD_RAW", os.path.join(OUT, "raw"))\nos.makedirs(OUT, exist_ok=True); os.makedirs(RAW, exist_ok=True)\nT0 = time.time()\nSTATS = {"small": SMALL, "stages": {}, "timings": {}}\nCC = "https://data.commoncrawl.org/"\nWARC = "crawl-data/CC-MAIN-2024-10/segments/1707947473347.0/warc/CC-MAIN-20240220211055-20240221001055-00000.warc.gz"\nWET = WARC.replace("/warc/", "/wet/").replace(".warc.gz", ".warc.wet.gz")\nHF = "https://huggingface.co/datasets/"\nC4_NOCLEAN = HF + "allenai/c4/resolve/main/en.noclean/c4-train.00000-of-07168.json.gz"\nEDU = HF + "HuggingFaceFW/fineweb-edu-llama3-annotations/resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet"\nARC = HF + "allenai/ai2_arc/resolve/refs%2Fconvert%2Fparquet/ARC-Easy/test/0000.parquet"\nHSW = HF + "Rowan/hellaswag/resolve/refs%2Fconvert%2Fparquet/default/validation/0000.parquet"\nWIKI_TR = HF + "Salesforce/wikitext/resolve/refs%2Fconvert%2Fparquet/wikitext-103-raw-v1/train/0000.parquet"\nWIKI_TE = HF + "Salesforce/wikitext/resolve/refs%2Fconvert%2Fparquet/wikitext-103-raw-v1/test/0000.parquet"\nTS_TR = HF + "roneneldan/TinyStories/resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet"\nTS_VA = HF + "roneneldan/TinyStories/resolve/refs%2Fconvert%2Fparquet/default/validation/0000.parquet"\nLID = "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin"\n\n\ndef log(*a):\n    print(round(time.time() - T0, 1), *a, flush=True)\n\n\ndef save():\n    STATS["timings"]["totalSec"] = round(time.time() - T0, 1)\n    with open(os.path.join(OUT, "ptd_prep_stats.json"), "w") as f:\n        json.dump(STATS, f, indent=1)\n\n\ndef dl(url, name, max_bytes=None, tries=10):\n    path = os.path.join(RAW, name)\n    if os.path.exists(path) and os.path.getsize(path) > 0:\n        return path\n    for t in range(tries):\n        try:\n            req = urllib.request.Request(url, headers={"User-Agent": "llt-ptd-prep/1.0"})\n            if max_bytes:\n                req.add_header("Range", "bytes=0-%d" % (max_bytes - 1))\n            with urllib.request.urlopen(req, timeout=120) as r, open(path + ".part", "wb") as f:\n                while True:\n                    chunk = r.read(1 << 20)\n                    if not chunk:\n                        break\n                    f.write(chunk)\n            os.replace(path + ".part", path)\n            log("downloaded", name, os.path.getsize(path))\n            return path\n        except Exception as e:  # Common Crawl answers SlowDown under load; back off and retry\n            log("download retry", t, name, repr(e)[:120])\n            time.sleep(min(300, 15 * 2 ** t))\n    raise RuntimeError("download failed " + url)\n\n\n# ---------------------------------------------------------------------------------------------\n# Workers (module level so multiprocessing can pickle them)\n_LID = None\n\n\ndef _alarm(signum, frame):\n    raise TimeoutError()\n\n\ndef w_extract(item):\n    url, html = item\n    import trafilatura\n    has_alarm = hasattr(signal, "SIGALRM")\n    if has_alarm:\n        signal.signal(signal.SIGALRM, _alarm); signal.alarm(5)\n    try:\n        txt = trafilatura.extract(html, favor_precision=True, include_comments=False, deduplicate=True)\n    except Exception:\n        txt = None\n    finally:\n        if has_alarm:\n            signal.alarm(0)\n    return url, txt or ""\n\n\ndef w_lang(text):\n    global _LID\n    if _LID is None:\n        import fasttext\n        fasttext.FastText.eprint = lambda x: None\n        _LID = fasttext.load_model(os.path.join(RAW, "lid.176.bin"))\n    # The low-level call: fasttext\'s predict() wrapper uses np.array(copy=False), which NumPy 2 rejects.\n    t = text.replace("\\n", " ")[:20000]\n    try:\n        prob, label = _LID.f.predict(t, 1, 0.0, "strict")[0]\n    except Exception:  # builds whose low-level binding differs (fasttext-numpy2)\n        labels, probs = _LID.predict(t, k=1)\n        label, prob = labels[0], probs[0]\n    return label.replace("__label__", ""), float(prob)\n\n\ndef w_filters(text):\n    """Gopher quality, Gopher repetition, C4. Returns (stage_failed, rule, cleaned_text)."""\n    r = ptd.gopher_quality(text)\n    if r:\n        return "gopher_quality", r, None\n    r = ptd.gopher_repetition(text)\n    if r:\n        return "gopher_repetition", r, None\n    cleaned, r = ptd.c4_clean(text)\n    if r:\n        return "c4", r, None\n    return None, None, cleaned\n\n\ndef w_c4_noterm(text):\n    _, r = ptd.c4_clean(text, terminal_punct=False)\n    return r\n\n\nA_HASH, B_HASH = ptd.make_hashes(112, seed=1)\n\n\ndef w_sig(text):\n    return ptd.minhash(ptd.shingles(text), A_HASH, B_HASH)\n\n\ndef w_datatrove(text):\n    try:\n        from datatrove.data import Document\n        from datatrove.pipeline.filters import GopherQualityFilter, GopherRepetitionFilter, C4QualityFilter\n        global _DT\n        try:\n            _DT\n        except NameError:\n            # C4 with the paper\'s numbers (3 sentences per page, 5 words per line), as ptd.c4_clean uses\n            _DT = (GopherQualityFilter(), GopherRepetitionFilter(), C4QualityFilter(min_num_sentences=3, min_words_per_line=5))\n        out = []\n        for f in _DT:\n            d = Document(text=text, id="x")\n            r = f.filter(d)\n            out.append(r is True)\n        return out\n    except Exception as e:\n        return repr(e)[:200]\n\n\ndef w_datasketch(text):\n    from datasketch import MinHash\n    m = MinHash(num_perm=112, seed=1)\n    w = ptd.norm_text(text).split()\n    for g in {" ".join(w[i:i + 5]) for i in range(max(1, len(w) - 4))}:\n        m.update(g.encode())\n    return m\n\n\ndef dedup(texts, pool):\n    """Exact dedup, then MinHash LSH (14 bands x 8 rows) over the survivors. Returns kept indices\n    and stats, including the exact Jaccard of every candidate pair."""\n    seen, keep_exact = {}, []\n    for i, t in enumerate(texts):\n        k = ptd.exact_key(t)\n        if k not in seen:\n            seen[k] = i; keep_exact.append(i)\n    sigs = np.stack(pool.map(w_sig, [texts[i] for i in keep_exact], chunksize=64))\n    pairs = ptd.lsh_candidates(sigs, 14, 8)\n    roots = ptd.clusters(len(keep_exact), pairs)\n    keep = [keep_exact[i] for i in range(len(keep_exact)) if roots[i] == i]\n    sample = sorted(pairs)[:3000]\n    js = [ptd.jaccard(ptd.shingles(texts[keep_exact[a]]), ptd.shingles(texts[keep_exact[b]])) for a, b in sample]\n    est = [float((sigs[a] == sigs[b]).mean()) for a, b in sample]\n    st = {"in": len(texts), "exactDuplicatesRemoved": len(texts) - len(keep_exact), "candidatePairs": len(pairs),\n          "nearDuplicatesRemoved": len(keep_exact) - len(keep), "out": len(keep),\n          "pairsChecked": len(sample), "pairJaccardMean": round(float(np.mean(js)), 4) if js else None,\n          "pairJaccardBelow0.5": int(sum(j < 0.5 for j in js)), "pairJaccardAtLeast0.8": int(sum(j >= 0.8 for j in js)),\n          "estVsExactMeanAbsErr": round(float(np.mean(np.abs(np.array(js) - np.array(est)))), 4) if js else None}\n    return keep, st, sigs, keep_exact, pairs\n\n\ndef short(t, n=400):\n    return re.sub(r"\\s+", " ", t)[:n]\n\n\n# ---------------------------------------------------------------------------------------------\ndef eval_sets():\n    import pandas as pd\n    arc = pd.read_parquet(dl(ARC, "arc_easy_test.parquet"))\n    hs = pd.read_parquet(dl(HSW, "hellaswag_val.parquet"))\n    arc_texts = [q + " " + " ".join(c["text"]) for q, c in zip(arc["question"], arc["choices"])]\n    hs_texts = [c + " " + e[int(l)] for c, e, l in zip(hs["ctx"], hs["endings"], hs["label"])]\n    STATS["decontam"] = {"arcEasyTestItems": len(arc_texts), "hellaswagValItems": len(hs_texts), "n": 13}\n    idx = ptd.build_eval_index(arc_texts + hs_texts, 13)\n    STATS["decontam"]["indexNgrams"] = len(idx)\n    return idx, hs\n\n\ndef part_a(pool, decon, score=None):\n    from warcio.archiveiterator import ArchiveIterator\n    t = time.time()\n    path = dl(CC + WARC, "warc.gz", max_bytes=(30 << 20) if SMALL else None)\n    n_rec, n_resp, items = 0, 0, []\n    try:\n        with open(path, "rb") as f:\n            for rec in ArchiveIterator(f):\n                n_rec += 1\n                if rec.rec_type != "response":\n                    continue\n                n_resp += 1\n                ctype = (rec.http_headers.get_header("Content-Type") or "") if rec.http_headers else ""\n                status = rec.http_headers.get_statuscode() if rec.http_headers else ""\n                if status != "200" or "html" not in ctype.lower():\n                    continue\n                body = rec.content_stream().read()\n                if len(body) > 2_000_000:\n                    continue\n                items.append((rec.rec_headers.get_header("WARC-Target-URI"), body.decode("utf-8", "replace")))\n    except Exception as e:  # a range download ends mid-record\n        log("warc read stopped", repr(e)[:100])\n    S = STATS["stages"]\n    S["warcRecords"] = n_rec; S["httpResponses"] = n_resp; S["html200"] = len(items)\n    STATS["source"] = {"warc": CC + WARC, "warcBytes": os.path.getsize(path), "wet": CC + WET,\n                       "licence": "Common Crawl Terms of Use (commoncrawl.org/terms-of-use); page text is copyright of each site"}\n    log("warc", n_rec, n_resp, len(items))\n    te = time.time()\n    ext = pool.map(w_extract, items, chunksize=16)\n    STATS["timings"]["extractSec"] = round(time.time() - te, 1)\n    del items\n    docs = [(u, x) for u, x in ext if x.strip()]\n    S["extractedNonEmpty"] = len(docs)\n    ex = {"extraction_empty": [u for u, x in ext if not x.strip()][:20]}\n    langs = pool.map(w_lang, [x for _, x in docs], chunksize=64)\n    lang_count = {}\n    for l, p in langs:\n        lang_count[l] = lang_count.get(l, 0) + 1\n    STATS["languagesTop"] = sorted(lang_count.items(), key=lambda kv: -kv[1])[:15]\n    en = [(u, x) for (u, x), (l, p) in zip(docs, langs) if l == "en" and p >= 0.65]\n    ex["not_english"] = [(u, l, round(p, 2), short(x, 160)) for (u, x), (l, p) in zip(docs, langs) if not (l == "en" and p >= 0.65)][:20]\n    S["english065"] = len(en)\n    res = pool.map(w_filters, [x for _, x in en], chunksize=32)\n    rules = {}\n    kept = []\n    for (u, x), (stage, rule, cleaned) in zip(en, res):\n        if stage:\n            rules.setdefault(stage, {}); rules[stage][rule] = rules[stage].get(rule, 0) + 1\n            ex.setdefault(stage, [])\n            if len(ex[stage]) < 40:\n                ex[stage].append((u, rule, short(x)))\n        else:\n            kept.append((u, cleaned))\n    STATS["ruleCounts"] = rules\n    S["afterGopherQuality"] = len(en) - sum(rules.get("gopher_quality", {}).values())\n    S["afterGopherRepetition"] = S["afterGopherQuality"] - sum(rules.get("gopher_repetition", {}).values())\n    S["afterC4"] = len(kept)\n    # C4 without the terminal-punctuation line rule, on the documents that reached the C4 stage\n    reach = [x for (u, x), (stage, _, _) in zip(en, res) if stage in (None, "c4")]\n    noterm = pool.map(w_c4_noterm, reach, chunksize=32)\n    S["afterC4_noTerminalPunctRule"] = S["afterGopherRepetition"] - sum(r is not None for r in noterm)\n    # Cross-check against datatrove\'s implementation of the same three filters\n    dt = pool.map(w_datatrove, [x for _, x in en], chunksize=32)\n    if dt and isinstance(dt[0], list):\n        ours_q = [ptd.gopher_quality(x) is None for _, x in en]\n        ours_r = [ptd.gopher_repetition(x) is None for _, x in en]\n        ours_c = [ptd.c4_clean(x)[1] is None for _, x in en]\n        STATS["datatroveAgreement"] = {"docs": len(en),\n                                       "gopherQuality": round(float(np.mean([a == d[0] for a, d in zip(ours_q, dt)])), 4),\n                                       "gopherRepetition": round(float(np.mean([a == d[1] for a, d in zip(ours_r, dt)])), 4),\n                                       "c4": round(float(np.mean([a == d[2] for a, d in zip(ours_c, dt)])), 4),\n                                       "datatroveKeepsAll3": int(sum(all(d) for d in dt)), "oursKeepsAll3": len(kept)}\n    else:\n        STATS["datatroveAgreement"] = {"error": str(dt[0]) if dt else "no docs"}\n    texts = [x for _, x in kept]\n    keep, dst, sigs, keep_exact, pairs = dedup(texts, pool)\n    STATS["funnelDedup"] = dst\n    S["afterExactDedup"] = dst["in"] - dst["exactDuplicatesRemoved"]\n    S["afterMinHashDedup"] = dst["out"]\n    roots = ptd.clusters(len(keep_exact), pairs)\n    groups = {}\n    for i, r in enumerate(roots):\n        groups.setdefault(r, []).append(keep_exact[i])\n    big = sorted(groups.values(), key=len, reverse=True)[:5]\n    ex["near_duplicate_clusters"] = [[(kept[i][0], short(kept[i][1], 200)) for i in g[:3]] + [len(g)] for g in big]\n    # datasketch on the same exact-deduplicated documents, same 14 x 8 banding\n    try:\n        from datasketch import MinHashLSH\n        ms = pool.map(w_datasketch, [texts[i] for i in keep_exact], chunksize=64)\n        lsh = MinHashLSH(num_perm=112, params=(14, 8))\n        dpairs = set()\n        for i, m in enumerate(ms):\n            for j in lsh.query(m):\n                dpairs.add((min(i, int(j)), max(i, int(j))))\n            lsh.insert(str(i), m)\n        droots = ptd.clusters(len(ms), dpairs)\n        STATS["datasketch"] = {"candidatePairs": len(dpairs), "nearDuplicatesRemoved": len(ms) - sum(1 for i, r in enumerate(droots) if r == i),\n                               "pairsInBoth": len(dpairs & pairs)}\n    except Exception as e:\n        STATS["datasketch"] = {"error": repr(e)[:200]}\n    final = [kept[i] for i in sorted(keep)]\n    dec = [ptd.contaminated(x, decon) for _, x in final]\n    ex["contaminated"] = [(u, short(x, 300)) for (u, x), c in zip(final, dec) if c][:20]\n    final = [d for d, c in zip(final, dec) if not c]\n    S["afterDecontamination"] = len(final)\n    n_e = n_i = n_docs = 0\n    scrubbed = []\n    for u, x in final:\n        y, a, b = ptd.scrub_pii(x)\n        n_e += a; n_i += b; n_docs += (a + b) > 0\n        scrubbed.append((u, y))\n    STATS["pii"] = {"emailsReplaced": n_e, "ipsReplaced": n_i, "docsChanged": n_docs, "docs": len(scrubbed)}\n    S["final"] = len(scrubbed)\n    if score is not None and scrubbed:\n        sc = score([x for _, x in scrubbed])\n        cut = STATS["classifier"]["tunedCut"]\n        STATS["funnelClassifier"] = {"atLeast3": int((sc >= 3).sum()), "atLeastTunedCut": int((sc >= cut).sum()),\n                                     "tunedCut": cut, "median": round(float(np.median(sc)), 3)}\n        order = np.argsort(sc)\n        ex["classifier_low"] = [(scrubbed[i][0], round(float(sc[i]), 2), short(scrubbed[i][1], 300)) for i in order[:15]]\n        ex["classifier_high"] = [(scrubbed[i][0], round(float(sc[i]), 2), short(scrubbed[i][1], 300)) for i in order[::-1][:15]]\n    STATS["funnelWords"] = {"final": int(sum(len(x.split()) for _, x in scrubbed)),\n                            "extracted": int(sum(len(x.split()) for _, x in docs))}\n    with open(os.path.join(OUT, "funnel_examples.json"), "w") as f:\n        json.dump(ex, f, indent=1)\n    with open(os.path.join(OUT, "funnel_final_docs.jsonl"), "w") as f:\n        for u, x in scrubbed:\n            f.write(json.dumps({"url": u, "text": x}) + "\\n")\n    STATS["timings"]["funnelSec"] = round(time.time() - t, 1)\n    save()\n    # The matching WET file: Common Crawl\'s own text extraction of the same pages\n    try:\n        from warcio.archiveiterator import ArchiveIterator\n        wpath = dl(CC + WET, "wet.gz", max_bytes=(10 << 20) if SMALL else None, tries=6)\n        wet = {}\n        try:\n            with open(wpath, "rb") as f:\n                for rec in ArchiveIterator(f):\n                    if rec.rec_type == "conversion":\n                        wet[rec.rec_headers.get_header("WARC-Target-URI")] = len(rec.content_stream().read().decode("utf-8", "replace").split())\n        except Exception as e:\n            log("wet read stopped", repr(e)[:100])\n        tw = {u: len(x.split()) for u, x in docs}\n        both = [u for u in tw if u in wet]\n        STATS["wetComparison"] = {"wetRecords": len(wet), "urlsInBoth": len(both),\n                                  "medianWordsTrafilatura": float(np.median([tw[u] for u in both])) if both else None,\n                                  "medianWordsWET": float(np.median([wet[u] for u in both])) if both else None}\n    except Exception as e:\n        STATS["wetComparison"] = {"error": repr(e)[:200]}\n    save()\n    return scrubbed\n\n\ndef part_b():\n    import pandas as pd\n    from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer\n    from sklearn.linear_model import Ridge\n    from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix\n    t = time.time()\n    df = pd.read_parquet(dl(EDU, "edu_ann_0000.parquet"), columns=["text", "score"])\n    if SMALL:\n        df = df.iloc[:6000]\n    rng = np.random.default_rng(0)\n    perm = rng.permutation(len(df))\n    n_test = len(df) // 6\n    te, tr = perm[:n_test], perm[n_test:]\n    hv = HashingVectorizer(n_features=2 ** 20, alternate_sign=False, ngram_range=(1, 2), norm=None)\n    tf = TfidfTransformer(sublinear_tf=True)\n    Xtr = tf.fit_transform(hv.transform(df["text"].iloc[tr]))\n    Xte = tf.transform(hv.transform(df["text"].iloc[te]))\n    ytr, yte = df["score"].values[tr], df["score"].values[te]\n    reg = Ridge(alpha=1.0).fit(Xtr, ytr)\n    p = reg.predict(Xte)\n    # A ridge regression shrinks toward the mean score, so a cut at 3.0 keeps almost nothing. Pick the\n    # cut on a slice of the TRAIN split (never on test), then report test F1 at that cut and AUC.\n    from sklearn.metrics import roc_auc_score\n    n_tune = len(tr) // 5\n    reg_t = Ridge(alpha=1.0).fit(Xtr[n_tune:], ytr[n_tune:])\n    pt = reg_t.predict(Xtr[:n_tune])\n    cuts = np.arange(1.5, 3.01, 0.05)\n    f1s = [f1_score(ytr[:n_tune] >= 3, pt >= c) for c in cuts]\n    best_cut = float(cuts[int(np.argmax(f1s))])\n    STATS["classifier"] = {"data": EDU, "rows": len(df), "train": len(tr), "test": len(te),\n                           "scoreCounts": {int(k): int(v) for k, v in df["score"].value_counts().sort_index().items()},\n                           "testMSE": round(float(np.mean((p - yte) ** 2)), 4),\n                           "testMSE_predictMean": round(float(np.mean((ytr.mean() - yte) ** 2)), 4),\n                           "f1At3": round(float(f1_score(yte >= 3, p >= 3)), 4),\n                           "precisionAt3": round(float(precision_score(yte >= 3, p >= 3, zero_division=0)), 4),\n                           "recallAt3": round(float(recall_score(yte >= 3, p >= 3)), 4),\n                           "confusionAt3": confusion_matrix(yte >= 3, p >= 3).tolist(),\n                           "positiveRateTest": round(float(np.mean(yte >= 3)), 4),\n                           "aucAt3": round(float(roc_auc_score(yte >= 3, p)), 4),\n                           "tunedCut": round(best_cut, 2),\n                           "f1AtTunedCut": round(float(f1_score(yte >= 3, p >= best_cut)), 4),\n                           "precisionAtTunedCut": round(float(precision_score(yte >= 3, p >= best_cut, zero_division=0)), 4),\n                           "recallAtTunedCut": round(float(recall_score(yte >= 3, p >= best_cut)), 4),\n                           "trainSec": round(time.time() - t, 1)}\n    log("classifier", STATS["classifier"])\n    save()\n\n    def score(texts):\n        return reg.predict(tf.transform(hv.transform(texts)))\n    return score\n\n\ndef doc_tokens(tok, texts, eot):\n    out = []\n    for i in range(0, len(texts), 2000):\n        for e in tok.encode_batch(texts[i:i + 2000]):\n            out.append(np.asarray(e.ids + [eot], dtype=np.uint16))\n    return out\n\n\ndef save_corpus(name, docs, cap=None):\n    lens = np.array([d.size for d in docs], dtype=np.int64)\n    if cap is not None:\n        n = int(np.searchsorted(np.cumsum(lens), cap)) + 1\n        docs, lens = docs[:n], lens[:n]\n    arr = np.concatenate(docs) if docs else np.zeros(0, np.uint16)\n    np.save(os.path.join(OUT, name + ".npy"), arr)\n    np.save(os.path.join(OUT, name + "_offsets.npy"), np.concatenate([[0], np.cumsum(lens)[:-1]]).astype(np.int64))\n    STATS.setdefault("corpora", {})[name] = {"docs": len(docs), "tokens": int(arr.size),\n                                             "meanDocTokens": round(float(lens.mean()), 1) if len(lens) else 0}\n    log("saved", name, len(docs), arr.size)\n\n\ndef part_c(pool, decon, score):\n    import pandas as pd\n    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders\n    t = time.time()\n    path = dl(C4_NOCLEAN, "c4noclean_00000.json.gz")\n    raw = []\n    with gzip.open(path, "rt", encoding="utf-8") as f:\n        for line in f:\n            raw.append(json.loads(line)["text"])\n            if SMALL and len(raw) >= 4000:\n                break\n    rng = random.Random(0)\n    rng.shuffle(raw)\n    n_hold = 400 if SMALL else 6000\n    hold, utrain = raw[:n_hold], raw[n_hold:]\n    C = STATS["c4"] = {"source": C4_NOCLEAN, "licence": "ODC-By 1.0 (allenai/c4), plus Common Crawl Terms of Use",\n                       "docs": len(raw), "bytes": int(sum(len(x.encode()) for x in raw)), "heldOutDocs": n_hold}\n    # Leakage between U\'s train and held-out split: MinHash over both, held-out docs with a candidate in train\n    sig_tr = np.stack(pool.map(w_sig, utrain, chunksize=64))\n    sig_ho = np.stack(pool.map(w_sig, hold, chunksize=64))\n    allsig = np.concatenate([sig_ho, sig_tr])\n    pairs = ptd.lsh_candidates(allsig, 14, 8)\n    leak = set()\n    for a, b in pairs:\n        if a < n_hold <= b and float((allsig[a] == allsig[b]).mean()) >= 0.8:\n            leak.add(a)\n    exact_tr = {ptd.exact_key(x) for x in utrain}\n    exact_leak = {i for i, x in enumerate(hold) if ptd.exact_key(x) in exact_tr}\n    C["leak"] = {"heldOut": n_hold, "nearDupInTrain_est0.8": len(leak), "exactDupInTrain": len(exact_leak),\n                 "leakedAny": len(leak | exact_leak)}\n    leak_idx = sorted(leak | exact_leak)\n    # Filtered corpus F: same stages as the funnel, from language ID on\n    langs = pool.map(w_lang, raw, chunksize=64)\n    en = [i for i, (l, p) in enumerate(langs) if l == "en" and p >= 0.65]\n    C["english065"] = len(en)\n    res = pool.map(w_filters, [raw[i] for i in en], chunksize=32)\n    rules = {}\n    passed, cleaned = [], {}\n    for i, (stage, rule, c) in zip(en, res):\n        if stage:\n            rules.setdefault(stage, {}); rules[stage][rule] = rules[stage].get(rule, 0) + 1\n        else:\n            passed.append(i); cleaned[i] = c\n    C["ruleCounts"] = rules; C["afterFilters"] = len(passed)\n    texts = [cleaned[i] for i in passed]\n    keep, dst, _, _, _ = dedup(texts, pool)\n    C["dedup"] = dst\n    fid = [passed[k] for k in sorted(keep)]\n    dec = [ptd.contaminated(cleaned[i], decon) for i in fid]\n    C["contaminatedRemoved"] = int(sum(dec))\n    fid = [i for i, d in zip(fid, dec) if not d]\n    ftext = {}\n    pe = pi = 0\n    for i in fid:\n        y, a, b = ptd.scrub_pii(cleaned[i]); pe += a; pi += b; ftext[i] = y\n    C["pii"] = {"emailsReplaced": pe, "ipsReplaced": pi}\n    C["finalDocs"] = len(fid)\n    C["finalBytes"] = int(sum(len(ftext[i].encode()) for i in fid))\n    hold_set = set(range(n_hold))\n    f_hold = [i for i in fid if i in hold_set]\n    f_train = [i for i in fid if i not in hold_set]\n    # Quality classifier scores on F (train side); E = top half\n    sc = score([ftext[i] for i in f_train]) if f_train else np.zeros(0)\n    cut = float(np.median(sc)) if len(sc) else 0.0\n    e_train = [i for i, s in zip(f_train, sc) if s >= cut]\n    C["classifierOnF"] = {"median": round(cut, 4), "fracAtLeast3": round(float(np.mean(sc >= 3)), 4) if len(sc) else None,\n                          "fracAtLeast2": round(float(np.mean(sc >= 2)), 4) if len(sc) else None, "eDocs": len(e_train)}\n    # Other domains\n    wtr = pd.read_parquet(dl(WIKI_TR, "wiki_train0.parquet"))["text"].tolist()\n    wte = pd.read_parquet(dl(WIKI_TE, "wiki_test.parquet"))["text"].tolist()\n\n    def articles(lines):\n        arts, cur = [], []\n        for ln in lines:\n            if re.match(r"^ = [^=].* = $", ln.rstrip("\\n")) and cur:\n                arts.append("".join(cur).strip()); cur = []\n            cur.append(ln)\n        if cur:\n            arts.append("".join(cur).strip())\n        return [a for a in arts if len(a) > 200]\n    wiki_tr, wiki_te = articles(wtr), articles(wte)\n    tstr = pd.read_parquet(dl(TS_TR, "ts_train0.parquet"))["text"].tolist()\n    tsva = pd.read_parquet(dl(TS_VA, "ts_val.parquet"))["text"].tolist()\n    if SMALL:\n        wiki_tr, tstr = wiki_tr[:300], tstr[:3000]\n    else:\n        tstr = tstr[:160000]\n    # Tokenizer: 4,096-token byte-level BPE on a fixed sample of the three domains\n    samp = utrain[: (2000 if SMALL else 12000)] + wiki_tr[: (100 if SMALL else 2500)] + tstr[: (2000 if SMALL else 60000)]\n    tok = Tokenizer(models.BPE())\n    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)\n    tok.decoder = decoders.ByteLevel()\n    trainer = trainers.BpeTrainer(vocab_size=4096, special_tokens=["<|eot|>"], initial_alphabet=pre_tokenizers.ByteLevel.alphabet())\n    tt = time.time(); tok.train_from_iterator(samp, trainer=trainer)\n    STATS["tokenizer"] = {"vocab": tok.get_vocab_size(), "sampleDocs": len(samp), "sampleBytes": int(sum(len(x.encode()) for x in samp)),\n                          "trainSec": round(time.time() - tt, 1)}\n    eot = tok.token_to_id("<|eot|>")\n    tok.save(os.path.join(OUT, "tokenizer.json"))\n    tb = np.array([len(tok.decode([i]).encode("utf-8")) if i != eot else 1 for i in range(tok.get_vocab_size())], dtype=np.int64)\n    np.save(os.path.join(OUT, "token_bytes.npy"), tb)\n    cap = 2_000_000 if SMALL else 60_000_000\n    save_corpus("U_train", doc_tokens(tok, utrain[: (3600 if SMALL else 40000)], eot), cap)\n    save_corpus("U_val_clean", doc_tokens(tok, [hold[i] for i in range(n_hold) if i not in set(leak_idx)], eot), 600_000)\n    save_corpus("U_val_leak", doc_tokens(tok, [hold[i] for i in leak_idx], eot), 600_000)\n    save_corpus("U_val", doc_tokens(tok, hold, eot), 600_000)\n    ftr_docs = doc_tokens(tok, [ftext[i] for i in f_train], eot)\n    save_corpus("F_train", ftr_docs)\n    fh_docs = doc_tokens(tok, [ftext[i] for i in f_hold], eot)\n    # Canaries: 160 held-out filtered documents with at least 100 tokens, first 100 tokens each\n    can_idx = [k for k, d in enumerate(fh_docs) if d.size >= 101][:160]\n    can = [fh_docs[k][:100] for k in can_idx]\n    np.save(os.path.join(OUT, "canaries.npy"), np.stack(can).astype(np.uint16) if can else np.zeros((0, 100), np.uint16))\n    rest = [d for k, d in enumerate(fh_docs) if k not in set(can_idx)]\n    STATS["canaries"] = {"count": len(can), "tokens": 100}\n    save_corpus("F_val", rest, 600_000)\n    save_corpus("E_train", doc_tokens(tok, [ftext[i] for i in e_train], eot))\n    save_corpus("wiki_train", doc_tokens(tok, wiki_tr, eot), cap)\n    save_corpus("wiki_test", doc_tokens(tok, wiki_te, eot), 600_000)\n    save_corpus("ts_train", doc_tokens(tok, tstr, eot), cap)\n    save_corpus("ts_val", doc_tokens(tok, tsva, eot), 600_000)\n    # HellaSwag validation, tokenized for the GPU kernel\n    hs = HS\n    items = []\n    for c, ends, lab in zip(hs["ctx"], hs["endings"], hs["label"]):\n        items.append({"ctx": tok.encode(c).ids, "endings": [tok.encode(" " + e).ids for e in ends], "label": int(lab)})\n    with open(os.path.join(OUT, "hellaswag_val_tok.json"), "w") as f:\n        json.dump(items, f)\n    STATS["timings"]["partCSec"] = round(time.time() - t, 1)\n    STATS["meta"] = {"eot": eot}\n    save()\n\n\nif __name__ == "__main__":\n    dl(LID, "lid.176.bin")\n    DECON, HS = eval_sets()\n    with Pool(int(os.environ.get("PTD_PROCS", "4"))) as pool:\n        score = part_b()\n        if "--skip-a" not in sys.argv:\n            part_a(pool, DECON, score)\n        part_c(pool, DECON, score)\n    save()\n    log("DONE")\n'
open('/tmp/prep_ptd.py', 'w').write(PREP_SRC)


In [ ]:

t = time.time()
env = dict(os.environ, PTD_RAW="/tmp/raw", PTD_PROCS="4")
p = subprocess.Popen([sys.executable, "/tmp/prep_ptd.py", "/kaggle/working"], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
p.wait()
print("prep exit", p.returncode, "wall", round(time.time() - t, 1))
assert p.returncode == 0


In [ ]:

import json
s = json.load(open("/kaggle/working/ptd_prep_stats.json"))
print(json.dumps({k: s[k] for k in ["stages", "c4", "classifier", "corpora", "tokenizer", "timings"] if k in s}, indent=1)[:12000])
for f in sorted(os.listdir("/kaggle/working")):
    print(f, os.path.getsize(os.path.join("/kaggle/working", f)))
